# 01 — Exploration FHIR : téléchargement, préfiltrage, échantillon

Première partie du pipeline BUPA (Conditions + MedicationRequest).

1. Télécharger et extraire les bundles FHIR synthétiques Synthea dans `data/raw/`
2. Préfiltrer en un seul passage (règles déterministes, sans LLM) dans `data/processed/filtered_conditions_medreq/`
3. Tirer l'échantillon de patients et inspecter les candidats

Ce notebook ne demande **ni GPU ni modèle**. Le code est dans `src/fhir.py`, les paramètres dans `config/settings.py`.

In [ ]:
# Uniquement sur Google Colab : décommenter pour récupérer le dépôt et installer les dépendances
# !git clone https://github.com/<ton-compte>/bupa.git
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Rend `config` et `src` importables depuis notebooks/ (ou depuis la racine du dépôt)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

## Paramètres de l'exécution

Toutes les valeurs par défaut sont dans `config/settings.py`. On peut les surcharger ici avant de lancer les étapes.
Le benchmark publié dans `results/` a été fait avec **40 patients**.

In [ ]:
# settings.N_PATIENTS = 40   # décommenter pour reproduire le benchmark publié
print("N_PATIENTS:", settings.N_PATIENTS, "| RANDOM_SEED:", settings.RANDOM_SEED)
print("PREFILTER_DROP_SUBSTRINGS:", settings.PREFILTER_DROP_SUBSTRINGS)

## 1) Télécharger et extraire Synthea

In [ ]:
from src import fhir

# fhir.reset_filtered_dataset()  # à décommenter si tu as modifié les règles PREFILTER_*
fhir.download_synthea()

## 2) Préfiltrage et dataset filtré (cache disque)

Règles conservatrices : on garde les conditions codées (SNOMED / ICD-10), on retire l'historique social
et les libellés SDOH évidents, puis on déduplique et on plafonne le nombre de candidats.

In [ ]:
df_index = fhir.build_filtered_dataset()
print("Filtered patients:", len(df_index))
display(df_index.head(10))
display(df_index[["n_conditions", "n_medication_requests"]].describe())

## 3) Échantillon et source préfiltrée (avant filtre LLM)

In [ ]:
df_sample, patients_prefilter, df_gold_prefilter = fhir.select_sample(df_index)
print("Prefiltered sample (candidates before LLM filtering)")
display(df_gold_prefilter)

In [ ]:
# Exemple de bundle préfiltré
example_pid = df_gold_prefilter.iloc[0]["patient_id"]
display(JSON(fhir.load_bundle(str(settings.FILTERED_DIR / f"{example_pid}.json"))))

**Suite :** `02_note_generation.ipynb` (gold source, génération des notes, reconstruction — nécessite MedGemma).